In [4]:
"""
TrollGuard – Full Python Code (Google Colab Version, using uploaded *_parsed_dataset.csv)

This version:
- Loads ALL *_parsed_dataset.csv files from
  /content/drive/MyDrive/TrollGuard_Project/datasets
- Uses Text as the text column and oh_label as the label column
- Normalises labels to binary 0/1
- Trains TF-IDF + Logistic Regression
- Exports predictions + artefacts
- Runs chat-export analysis
"""

# ==============================================
# 0. Setup and Imports
# ==============================================

# Before running this cell, make sure you have:
# from google.colab import drive
# drive.mount('/content/drive')

import os
import re
import glob
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

sns.set(style="whitegrid")

# ==============================================
# 1. Configuration
# ==============================================

# Auto-detect: Colab/Drive vs local
if os.path.exists("/content/drive/MyDrive/TrollGuard_Project"):
    BASE_DIR = "/content/drive/MyDrive/TrollGuard_Project"
elif os.path.exists("/content/drive/MyDrive/TrollGuard_Project_Gloryia_2025"):
    BASE_DIR = "/content/drive/MyDrive/TrollGuard_Project_Gloryia_2025"
else:
    BASE_DIR = os.path.dirname(os.path.abspath("."))
    if "ipynb_file" in os.getcwd():
        BASE_DIR = os.path.dirname(os.path.dirname(os.getcwd()))

for name in ("datasets", "dataset"):
    _p = os.path.join(BASE_DIR, name)
    if os.path.isdir(_p):
        DATASETS_DIR = _p
        break
else:
    DATASETS_DIR = os.path.join(BASE_DIR, "dataset")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
ARTEFACT_DIR = OUTPUT_DIR

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ARTEFACT_DIR, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DATASETS_DIR:", DATASETS_DIR)

# ==============================================
# 2. Data Loading (ALL uploaded *_parsed_dataset.csv)
# ==============================================

pattern = os.path.join(DATASETS_DIR, "*_parsed_dataset.csv")
csv_files = sorted(glob.glob(pattern))

print("\nFound CSV files:")
for p in csv_files:
    print(" -", os.path.basename(p))

if not csv_files:
    raise FileNotFoundError(f"No *_parsed_dataset.csv files found in {DATASETS_DIR}")

dfs = []

for path in csv_files:
    fname = os.path.basename(path)
    print(f"\nLoading: {fname}")
    tmp = pd.read_csv(path)
    print("  Columns:", list(tmp.columns))

    # Your files all have Text + oh_label; use those.
    if "Text" not in tmp.columns or "oh_label" not in tmp.columns:
        print("  Skipping – does not have Text + oh_label")
        continue

    # Keep only the two columns and rename to standard names
    tmp = tmp[["Text", "oh_label"]].rename(columns={"Text": "text", "oh_label": "label"})
    dfs.append(tmp)
    print(f"  Using columns: Text -> text, oh_label -> label")
    print(f"  Rows loaded from this file: {len(tmp)}")

if not dfs:
    raise RuntimeError("No dataset could be normalised to [text, label]. "
                       "Check that your CSVs have Text and oh_label columns.")

df = pd.concat(dfs, ignore_index=True)
print("\nCombined dataset shape:", df.shape)
print(df.head())

# ==============================================
# 2.1 Normalise labels to binary 0/1
# ==============================================

print("\nOriginal label value counts:")
print(df["label"].value_counts(dropna=False))

# Try to convert label to numeric; if it fails, fallback to string mapping
try:
    df["label"] = pd.to_numeric(df["label"])
    print("\nLabel converted to numeric.")
except Exception:
    print("\nLabel not purely numeric. Converting via simple mapping.")
    NON_BULLY_VALUES = {
        "none",
        "normal",
        "non-toxic",
        "non_toxic",
        "non-aggressive",
        "not_cyberbullying",
        "safe",
        "neutral"
    }
    def to_binary_label(x):
        s = str(x).strip().lower()
        if s in NON_BULLY_VALUES or s == "0":
            return 0
        if s == "1":
            return 1
        return 1
    df["label"] = df["label"].apply(to_binary_label)

# If numeric but not 0/1, squash anything >0 to 1
if df["label"].dtype.kind in "iu":
    uniq = set(df["label"].unique())
    if not uniq <= {0, 1}:
        df["label"] = df["label"].apply(lambda v: 0 if v == 0 else 1)

print("\nLabel value counts AFTER normalisation:")
print(df["label"].value_counts(dropna=False))

# Drop missing and keep only text + label
df = df[["text", "label"]].dropna()
print("\nAfter dropping NA, shape:", df.shape)

# ==============================================
# 3. Text Cleaning Function
# ==============================================

def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"[^a-z ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("\nApplying cleaning function...")
df["clean_text"] = df["text"].apply(clean_text)
print(df[["text", "clean_text"]].head())

# ==============================================
# 4. EDA
# ==============================================

plt.figure(figsize=(5, 4))
ax = sns.countplot(x="label", data=df)
ax.set_title("Class Distribution (0 = non-bullying, 1 = bullying)")
plt.tight_layout()
class_dist_path = os.path.join(OUTPUT_DIR, "class_distribution.png")
plt.savefig(class_dist_path)
plt.close()
print("Saved class distribution plot to:", class_dist_path)

df["text_length"] = df["clean_text"].apply(lambda x: len(x.split()))

plt.figure(figsize=(6, 4))
ax = sns.histplot(data=df, x="text_length", bins=30, hue="label", kde=False)
ax.set_title("Message Length Distribution by Class")
plt.tight_layout()
length_hist_path = os.path.join(OUTPUT_DIR, "length_distribution.png")
plt.savefig(length_hist_path)
plt.close()
print("Saved length distribution plot to:", length_hist_path)

summary_by_label = df.groupby("label")["text_length"].describe()
print("\nText length summary by label:")
print(summary_by_label)

# ==============================================
# 5. Train–Test Split
# ==============================================

X = df["clean_text"].values
y = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Test samples:", len(X_test))

# ==============================================
# 6. TF-IDF + Model
# ==============================================

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

print("\nFitting TF-IDF on training data...")
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

print("TF-IDF matrix shape (train):", X_train_vec.shape)

model = LogisticRegression(max_iter=300)
print("Training Logistic Regression model...")
model.fit(X_train_vec, y_train)

# ==============================================
# 7. Evaluation
# ==============================================

y_pred = model.predict(X_test_vec)

acc = accuracy_score(y_test, y_pred)
print("\nAccuracy:", acc)

print("\nClassification report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:")
print(cm)

plt.figure(figsize=(4, 3))
ax = sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Confusion Matrix")
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
plt.savefig(cm_path)
plt.close()
print("Saved confusion matrix plot to:", cm_path)

# ==============================================
# 8. Save Artefacts + Predictions
# ==============================================

vectoriser_path = os.path.join(ARTEFACT_DIR, "tfidf.joblib")
model_path = os.path.join(ARTEFACT_DIR, "logreg_model.joblib")

joblib.dump(tfidf, vectoriser_path)
joblib.dump(model, model_path)

print("\nSaved TF-IDF vectoriser to:", vectoriser_path)
print("Saved Logistic Regression model to:", model_path)

results_df = pd.DataFrame({
    "text": X_test,
    "true_label": y_test,
    "predicted_label": y_pred
})

predictions_path = os.path.join(OUTPUT_DIR, "predictions_dataset.csv")
results_df.to_csv(predictions_path, index=False)
print("Saved dataset-level predictions to:", predictions_path)

# ==============================================
# 9. Chat Export Analysis
# ==============================================

from datetime import datetime

CHAT_FILE = os.path.join(DATASETS_DIR, "sample_chat.txt")

if os.path.exists(CHAT_FILE):
    print("\nLoading chat export from:", CHAT_FILE)
else:
    print("\nChat export file not found. Place a file at:", CHAT_FILE)

def parse_whatsapp_chat(chat_path: str):
    rows = []
    if not os.path.exists(chat_path):
        return pd.DataFrame(columns=["timestamp", "sender", "message_text"])

    with open(chat_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                date_part, rest = line.split(" - ", 1)
                ts = datetime.strptime(date_part, "%d/%m/%Y, %H:%M")
                sender_part, msg = rest.split(": ", 1)
                rows.append({
                    "timestamp": ts,
                    "sender": sender_part,
                    "message_text": msg
                })
            except ValueError:
                continue

    return pd.DataFrame(rows)

chat_df = parse_whatsapp_chat(CHAT_FILE)
print("Parsed chat shape:", chat_df.shape)
print(chat_df.head())

if not chat_df.empty:
    chat_df["clean_text"] = chat_df["message_text"].apply(clean_text)

    if "tfidf" not in globals() or "model" not in globals():
        tfidf = joblib.load(vectoriser_path)
        model = joblib.load(model_path)

    X_chat = tfidf.transform(chat_df["clean_text"])
    chat_df["bullying_label"] = model.predict(X_chat)

    chat_pred_path = os.path.join(OUTPUT_DIR, "predictions_chat_export.csv")
    chat_df.to_csv(chat_pred_path, index=False)
    print("Saved chat-level predictions to:", chat_pred_path)

    summary = chat_df.groupby("sender")["bullying_label"].agg(["count", "sum"]).reset_index()
    summary.rename(columns={"count": "total_messages", "sum": "bullying_messages"}, inplace=True)
    summary["bullying_rate"] = summary["bullying_messages"] / summary["total_messages"]

    sender_summary_path = os.path.join(OUTPUT_DIR, "chat_user_summary.csv")
    summary.to_csv(sender_summary_path, index=False)
    print("Saved sender summary to:", sender_summary_path)
else:
    print("No chat records parsed. Chat analysis section did not run.")



BASE_DIR: /content/drive/MyDrive/TrollGuard_Project
DATASETS_DIR: /content/drive/MyDrive/TrollGuard_Project/datasets

Found CSV files:
 - aggression_parsed_dataset.csv
 - attack_parsed_dataset.csv
 - kaggle_parsed_dataset.csv
 - toxicity_parsed_dataset.csv
 - twitter_parsed_dataset.csv
 - twitter_racism_parsed_dataset.csv
 - twitter_sexism_parsed_dataset.csv
 - youtube_parsed_dataset.csv

Loading: aggression_parsed_dataset.csv
  Columns: ['index', 'Text', 'ed_label_0', 'ed_label_1', 'oh_label']
  Using columns: Text -> text, oh_label -> label
  Rows loaded from this file: 115864

Loading: attack_parsed_dataset.csv
  Columns: ['index', 'Text', 'ed_label_0', 'ed_label_1', 'oh_label']
  Using columns: Text -> text, oh_label -> label
  Rows loaded from this file: 115864

Loading: kaggle_parsed_dataset.csv
  Columns: ['index', 'oh_label', 'Date', 'Text']
  Using columns: Text -> text, oh_label -> label
  Rows loaded from this file: 8799

Loading: toxicity_parsed_dataset.csv
  Columns: ['ind